# Ekegusii Data Preparation: Local Corpora, Three-Part Test Set, and Curriculum Construction

**Sources:** 6 local raw corpora (not the published `aykgeh/Ekegusii-English-Kiswahili-Parallel-Corpus`
HF dataset that the original `00_data_prep.ipynb` used):

| File | Content |
|---|---|
| `Bible_engswguz (2).csv` | Scripture (3 non-overlapping translations) + 135 Anansi folktale rows mislabeled into the same file |
| `MixGenre_EngSwaGuz (1).csv` | `lughayangu` (dictionary/phrasebook) + `african_storybook` (storybook, cleaner punctuation than the Bible file's copy) |
| `kenyan_psa_multilingual_dataset...Final (6).csv` | PSA content, 73% LLM-generated/synthetic, 27% real |
| `master_sentence_EgSwGuzcorpus (5).csv` | An older aggregation, ~97% redundant with the other 5 files once deduped |
| `psa_ke_test.csv` / `psa_ke_train.csv` | Native-speaker-collected PSAs, domain-labeled (Agriculture/Education/Governance/Health/Security) |

**Purpose of this notebook.** This is `00_data_prep.ipynb` for this iteration of the project - the
first notebook to run, before EDA and the training experiments. It produces the same file contract
`data_io.py` already expects (`stage1.jsonl`, `stage2.jsonl`, `mixed.jsonl`,
`validation_general.jsonl`, `validation_psa.jsonl`, `test.jsonl`, the flat `.csv` equivalents,
`whole_dataset.csv`/`.parquet`, `mixture.json`), so `nllb_training.ipynb` / `mt5_training.ipynb`
need no changes to consume it.

**What's identical to the original `00_data_prep.ipynb`:** the Stage 1 / Stage 2 / Mixed curriculum
algorithm (PSA rows upsampled x4, a 25%-of-Stage-2 replay slice of Stage 1 to resist forgetting),
the 2-60 word length gate on held-out sets only, the leakage check against train, and the fixed
seed (42) governing every stochastic step.

**What's different, and why:**

1. **Source is the 6 local files, not the HF dataset** - loaded directly here rather than assumed
   already merged, so this notebook also has to decide encoding, de-duplication, and category
   labels (`PSA_non_PSA`, `psa_domain`, `category`) that the HF dataset came with pre-attached.
2. **Test construction is a three-part design**, replacing the old single random-10%-PSA slice:
   - **PSA test**: 2,000 rows, 400 per domain (Agriculture/Education/Governance/Health/Security),
     drawn only from the native-speaker `psa_ke_test`/`psa_ke_train` files.
   - **Bible test**: 2,000 random rows, scripture only (folktales excluded).
   - **General Ekegusii test**: four independent slices, each a percentage of *that source's own
     remaining pool* after the first two tests are removed - 5% of the scripture pool, 25% of the
     storybook pool, 10% of the PSA pool (this one *does* include the synthetic `kenyan_psa` data),
     25% of the dictionary/`lughayangu` pool. No fixed total; size is whatever the math produces.

   Built in that exact order - PSA, then Bible, then general - each removed from the shared pool
   before the next is constructed, so the three test sets and validation never overlap.

**Scope and limitations** (same spirit as the original notebook's disclosure):

1. **De-duplication is exact-match only**, on a case-folded (English, Ekegusii) pair. It does
   *not* catch same-verse duplicates where the English differs across the Bible file's two
   translation sub-sources (`Trilingual-Bible` vs `bible_ebible`) - Section 8's leakage check is
   what catches that (see the note in Section 8 for a real example).
2. **`PSA_non_PSA` and `psa_domain` are derived here**, not given, since the raw files don't carry
   an HF-dataset-style label column - see Section 3 for exactly how.
3. **Encoding is repaired, not assumed.** `Bible_engswguz` and `master_sentence` mix UTF-8 with
   stray CP1252 bytes (Windows smart quotes never reconciled to UTF-8 before the files were
   merged). See Section 2.

## Table of contents

1. Environment setup
2. Robust CSV loading (encoding repair)
3. Loading and normalizing the 6 raw corpora
4. Shuffling and de-duplication
5. Three-part test construction: PSA -> Bible -> general Ekegusii
6. Validation decomposition (general vs. PSA)
7. Curriculum construction: Stage 1, Stage 2, Mixed
8. Held-out set construction: length gating and leakage verification
9. Serialization and export
10. Cross-architecture portability notes
11. Reproducibility summary

## 1. Environment setup

**Dependencies.** Only `pandas`, `numpy`, and `pyarrow` (for the `whole_dataset.parquet` export) -
no `scikit-learn` here; the one place the original notebook used
`sklearn.model_selection.train_test_split(..., stratify=...)`, this notebook uses a small
hand-rolled `stratified_split()` instead (sample a fraction from each stratum independently),
since the local environment this was developed against didn't have scikit-learn installed and the
stratification logic is a few lines either way.

**Project root.** Same convention as every other notebook in this project: walk up from the
current directory looking for `arch_config.py`, so this works whether the notebook sits at the
project root or in a `notebooks/` subfolder.

In [32]:
import sys, os, re, io, json, codecs
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root(marker="arch_config.py"):
    current = Path.cwd().resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not find {marker} by walking up from {current}")


PROJECT_ROOT = find_project_root()
print(f"project root: {PROJECT_ROOT}")

# Where the 6 raw CSVs live. Upload them here (or point this at wherever you put them)
# before running Section 3.
RAW_DATA_DIR = PROJECT_ROOT / "raw_data"
if not RAW_DATA_DIR.exists():
    print(f"NOTE: {RAW_DATA_DIR} doesn't exist yet - create it and upload the 6 raw CSVs there, "
          f"or change RAW_DATA_DIR above to wherever you put them.")

# Matches DATA_DIR in arch_config.py - this is what lets nllb_training.ipynb / mt5_training.ipynb
# load this notebook's output with zero path configuration of their own.
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)
print(f"raw data from : {RAW_DATA_DIR}")
print(f"output to     : {DATA_DIR}")

SEED = 42
ENG, SWH, GUZ = "eng_Latn", "swh_Latn", "guz_Latn"   # NLLB-style codes, matches train.py

PSA_UPSAMPLE = 4          # repetition factor applied to PSA rows in Stage 2 / Mixed
REPLAY_FRACTION = 0.25    # share of Stage 2 drawn back from Stage 1, to resist forgetting
MIN_WORDS, MAX_WORDS = 2, 60   # length gate - applied to evaluation sets only, never to train

PSA_TEST_TOTAL = 2000
PSA_TEST_PER_DOMAIN = 400
BIBLE_TEST_TOTAL = 2000
VALIDATION_FRAC = 0.10

GENERAL_TEST_FRACS = {"bible": 0.05, "storybook": 0.25, "psa": 0.10, "dictionary": 0.25}

DOMAIN_NORMALIZE = {"Security & Safety": "Security"}   # kenyan_psa's label -> psa_ke's label
CANONICAL_DOMAINS = ["Agriculture", "Education", "Governance", "Health", "Security"]


def normalize_domain(d):
    if pd.isna(d):
        return d
    return DOMAIN_NORMALIZE.get(d, d)


project root: /home/jovyan/PSA/Final_Training
raw data from : /home/jovyan/PSA/Final_Training/raw_data
output to     : /home/jovyan/PSA/Final_Training/data


## 2. Robust CSV loading (encoding repair)

### Method
Read each file as raw bytes and decode as UTF-8, but register a custom error handler that repairs
only the invalid byte spans by re-decoding *just those bytes* as CP1252, rather than falling back
to a single encoding for the whole file.

### Rationale
`Bible_engswguz` and `master_sentence` fail a plain UTF-8 read (`'utf-8' codec can't decode byte
0x93 ... invalid start byte`). The tempting fix - reading the whole file as `latin1`, which never
raises since it maps every byte 0-255 to *some* character - is wrong: it silently mangles every
genuine multi-byte UTF-8 character in the file too (a curly apostrophe's 3-byte UTF-8 sequence
becomes 3 garbled Latin-1 characters instead of one correct one). The actual corruption is a
*mix*: most of the file is valid UTF-8 (confirmed - a blanket-Latin1 read of this same file
produces textbook UTF-8-decoded-as-Latin1 mojibake for its curly quotes), but a handful of Windows
smart-quote characters were typed/pasted in CP1252 and never converted before the file was saved,
so isolated single bytes in the CP1252 punctuation range (`0x91`-`0x97`, etc.) sit inline within
otherwise-correct UTF-8 text. Decoding UTF-8 with a CP1252 fallback *only on the bytes that
actually fail* repairs exactly that mix without touching anything that was already correct.

A separate, smaller issue: 5 byte values (`0x81`, `0x8D`, `0x8F`, `0x90`, `0x9D`) are genuinely
undefined in CP1252, so even this repair can't recover a real character for them - Python's
`cp1252` codec maps them to the equivalent C1 control code as a last resort rather than raising.
These show up in ~35 rows out of 87k (0.04%) as unrecoverable stray characters predating this
pipeline; `clean_text()` below strips them since they're never legitimate content.

In [33]:
def _cp1252_fallback(err):
    bad = err.object[err.start:err.end]
    return (bad.decode("cp1252", errors="replace"), err.end)


codecs.register_error("cp1252_fallback", _cp1252_fallback)


def read_csv_robust(path, **kwargs):
    with open(path, "rb") as fh:
        raw = fh.read()
    text = raw.decode("utf-8", errors="cp1252_fallback")
    return pd.read_csv(io.StringIO(text), **kwargs)


_C1_CONTROL_RE = re.compile(r"[\x80-\x9f]")


def clean_text(s):
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    if not s or s.lower() == "nan":
        return np.nan
    s = _C1_CONTROL_RE.sub("", s)
    s = s.strip()
    return s if s else np.nan


## 3. Loading and normalizing the 6 raw corpora

### Method
Each file has its own column names and its own way of indicating PSA-vs-not; this section maps
all 6 onto one shared schema: `English`, `Kiswahili`, `Ekegusii`, `dataset_origin`, `PSA_non_PSA`,
`psa_domain`, `category`, `provenance`, `orig_source_file`, `orig_id`, `quality_flag`.

### Rationale for the per-file labeling decisions

- **`Bible_engswguz`**: its `source` column has 4 values. Three are genuine scripture
  (`Trilingual-Bible`, `bible_ebible`, `ENG-EKE` - verified to have zero verse overlap with each
  other, so they're complementary, not duplicated). The fourth, `Trilingual-Stories` (135 rows),
  is Anansi folktale content, not scripture - it's labeled `category="storybook"`, not
  `"scripture"`, specifically so it doesn't leak into the Bible test set in Section 5.
- **`MixGenre_EngSwaGuz`**: `lughayangu` (316 rows, short everyday sentences - confirmed this is
  the "dictionary" data) and `african_storybook` (95 rows - the same Anansi content as
  `Bible_engswguz`'s `Trilingual-Stories`, but with correct punctuation: apostrophes and curly
  quotes intact, vs. the Bible file's copy, which has apostrophes stripped from the English side).
- **`kenyan_psa_multilingual_dataset`**: every row is PSA. `Domain` is normalized
  (`"Security & Safety"` -> `"Security"`, matching `psa_ke`'s label) and `Provenance`
  (`real`/`synthetic`) is kept as `provenance`, since 73% of this file is LLM-generated.
- **`master_sentence_EgSwGuzcorpus`**: an older aggregation. Rows with no Ekegusii translation
  (2,086 of them - English+Kiswahili only) are dropped outright; they can't produce an
  Ekegusii-target training example. The rest get the same scripture/storybook/psa labeling as
  above based on its own `source` column.
- **`psa_ke_test`/`psa_ke_train`**: native-speaker-collected PSAs, domain-labeled. `quality_flags`
  and `is_psa_register==False` rows are tagged `quality_flag="flagged"` here rather than dropped -
  they're excluded from test-set candidate pools in Section 5, but kept available for training.

In [34]:
CORE_COLS = ["English", "Kiswahili", "Ekegusii", "dataset_origin", "PSA_non_PSA",
             "psa_domain", "category", "provenance", "orig_source_file", "orig_id",
             "quality_flag"]


def load_bible():
    path = RAW_DATA_DIR / "Bible_engswguz (2).csv"
    df = read_csv_robust(path)
    df["English"] = df["english"].map(clean_text)
    df["Kiswahili"] = df["swahili"].map(clean_text)
    df["Ekegusii"] = df["ekegusii"].map(clean_text)
    df["orig_id"] = df["id"].astype(str)
    is_story = df["source"] == "Trilingual-Stories"
    df["dataset_origin"] = df["source"]
    df["PSA_non_PSA"] = "Non-PSA"
    df["psa_domain"] = np.nan
    df["category"] = np.where(is_story, "storybook", "scripture")
    df["provenance"] = "real"
    df["orig_source_file"] = "Bible_engswguz"
    df["quality_flag"] = np.nan
    print(f"  bible          : {len(df):>7,} rows  (scripture={int((~is_story).sum()):,}, "
          f"storybook={int(is_story.sum()):,})")
    return df[CORE_COLS]


def load_mixgenre():
    path = RAW_DATA_DIR / "MixGenre_EngSwaGuz (1).csv"
    df = read_csv_robust(path)
    df["English"] = df["english"].map(clean_text)
    df["Kiswahili"] = df["swahili"].map(clean_text)
    df["Ekegusii"] = df["ekegusii"].map(clean_text)
    df["orig_id"] = df["id"].astype(str)
    df["dataset_origin"] = df["source"]
    df["PSA_non_PSA"] = "Non-PSA"
    df["psa_domain"] = np.nan
    df["category"] = np.where(df["source"] == "lughayangu", "dictionary", "storybook")
    df["provenance"] = "real"
    df["orig_source_file"] = "MixGenre_EngSwaGuz"
    df["quality_flag"] = np.nan
    print(f"  mixgenre       : {len(df):>7,} rows  "
          f"(dictionary/lughayangu={int((df['category']=='dictionary').sum()):,}, "
          f"storybook={int((df['category']=='storybook').sum()):,})")
    return df[CORE_COLS]


def load_kenyan_psa():
    path = RAW_DATA_DIR / "kenyan_psa_multilingual_dataset - kenyan_psa_multilingual_dataset_Final (6).csv"
    df = read_csv_robust(path)
    df["English"] = df["English"].map(clean_text)
    df["Kiswahili"] = df["Kiswahili"].map(clean_text)
    df["Ekegusii"] = df["Ekegusii"].map(clean_text)
    df["orig_id"] = df["PSA_ID"].astype(str)
    df["dataset_origin"] = "kenyan_psa_multilingual"
    df["PSA_non_PSA"] = "PSA"
    df["psa_domain"] = df["Domain"].map(normalize_domain)
    df["category"] = "psa"
    df["provenance"] = df["Provenance"].fillna("unknown")
    df["orig_source_file"] = "kenyan_psa_multilingual_dataset"
    df["quality_flag"] = np.nan
    print(f"  kenyan_psa     : {len(df):>7,} rows  "
          f"(real={int((df['provenance']=='real').sum()):,}, "
          f"synthetic={int((df['provenance']=='synthetic').sum()):,})")
    return df[CORE_COLS]


def load_master():
    path = RAW_DATA_DIR / "master_sentence_EgSwGuzcorpus (5).csv"
    df = read_csv_robust(path)
    df["English"] = df["English"].map(clean_text)
    df["Kiswahili"] = df["Kiswahili"].map(clean_text)
    df["Ekegusii"] = df["Ekegusii"].map(clean_text)
    before = len(df)
    df = df[df["Ekegusii"].notna()].reset_index(drop=True)
    dropped = before - len(df)
    df["orig_id"] = df["concept_id"].astype(str)
    df["dataset_origin"] = df["source"]
    scripture = df["source"].isin(["Trilingual-Bible", "ENG-EKE"])
    storybook = df["source"].isin(["Trilingual-Stories", "Trilingual-Storybooks"])
    psa = df["source"] == "PSA"
    df["PSA_non_PSA"] = np.where(psa, "PSA", "Non-PSA")
    df["psa_domain"] = np.nan
    df["category"] = np.select([scripture, storybook, psa], ["scripture", "storybook", "psa"],
                                default="other")
    df["provenance"] = "real"
    df["orig_source_file"] = "master_sentence_EgSwGuzcorpus"
    df["quality_flag"] = np.nan
    print(f"  master         : {len(df):>7,} rows  (dropped {dropped:,} EN-SW-only rows with no "
          f"Ekegusii translation)")
    return df[CORE_COLS]


def load_psa_ke(name):
    path = RAW_DATA_DIR / f"{name}.csv"
    df = read_csv_robust(path)
    df["English"] = df["english"].map(clean_text)
    df["Kiswahili"] = df["kiswahili"].map(clean_text)
    df["Ekegusii"] = df["ekegusii"].map(clean_text)
    df["orig_id"] = df["psa_id"].astype(str)
    df["dataset_origin"] = name
    df["PSA_non_PSA"] = "PSA"
    df["psa_domain"] = df["domain"].map(normalize_domain)
    df["category"] = "psa"
    df["provenance"] = "real"
    df["orig_source_file"] = name
    flagged = df["quality_flags"].notna() | (df["is_psa_register"] == False)  # noqa: E712
    df["quality_flag"] = pd.Series(np.where(flagged, "flagged", None), dtype=object)
    print(f"  {name:15s}: {len(df):>7,} rows  (flagged={int(flagged.sum()):,})")
    return df[CORE_COLS]


print("Loading and normalizing the 6 raw corpora")
print("-" * 70)
loaders = [load_psa_ke("psa_ke_test"), load_psa_ke("psa_ke_train"),
           load_mixgenre(), load_bible(), load_kenyan_psa(), load_master()]
# Concatenation order controls which copy "wins" the later exact-dedupe (Section 4):
# native-speaker PSA and MixGenre's cleaner-punctuation storybook/dictionary rows are
# listed first, master_sentence last (it's the most redundant source).
df = pd.concat(loaders, ignore_index=True)
print(f"\nConcatenated: {len(df):,} rows before de-duplication")


Loading and normalizing the 6 raw corpora
----------------------------------------------------------------------
  psa_ke_test    :     573 rows  (flagged=0)
  psa_ke_train   :   3,509 rows  (flagged=262)
  mixgenre       :     411 rows  (dictionary/lughayangu=316, storybook=95)
  bible          :  65,547 rows  (scripture=65,412, storybook=135)
  kenyan_psa     :  20,507 rows  (real=5,510, synthetic=14,984)
  master         :  38,610 rows  (dropped 2,086 EN-SW-only rows with no Ekegusii translation)

Concatenated: 129,157 rows before de-duplication


## 4. Shuffling and de-duplication

### Method
De-duplicate first (exact, case-folded (English, Ekegusii) match, keep first occurrence), *then*
shuffle - the reverse order from the original HF-based notebook.

### Rationale
The original notebook shuffled before de-duplicating, so which copy of a duplicate survived was
arbitrary (whichever landed first after a random shuffle). Here, the concatenation order from
Section 3 is deliberately meaningful - native-speaker PSA and MixGenre's cleaner-punctuation
storybook/dictionary rows are listed first, `master_sentence` last - so de-duplicating before
shuffling makes "which copy survives" a quality-driven choice instead of a random one: MixGenre's
correctly-punctuated storybook rows win over the Bible file's and master's copies (which have
apostrophes stripped from the English side), and native-speaker PSA content wins over
`master_sentence`'s older copy of the same PSAs. The shuffle still happens afterward, for the same
reason the original notebook did it: the 6 files are concatenated in contiguous blocks, and any
downstream partitioning on unshuffled data risks pulling a whole contiguous source block into one
split.

A fresh integer `concept_id` is assigned post-dedup, since the 6 raw files use incompatible ID
schemes (plain integers, `"M000001"`-style, `"HEA_00001"`-style) that can't be merged as-is.

In [35]:
print("De-duplication")
print("-" * 70)
before = len(df)
dedupe_key = (df["English"].astype(str).str.strip().str.lower() + "\x1f"
              + df["Ekegusii"].astype(str).str.strip().str.lower())
df = df[~dedupe_key.duplicated(keep="first")].reset_index(drop=True)
print(f"Removed {before - len(df):,} exact duplicate (English, Ekegusii) pairs.")
print(f"Rows remaining: {len(df):,}")

psa_mask = df["PSA_non_PSA"] == "PSA"
print(f"\nPSA rows      : {int(psa_mask.sum()):,}")
print(f"Non-PSA rows  : {int((~psa_mask).sum()):,}")
print("\nRows by category:")
print(df["category"].value_counts().to_string())
print("\nRows by orig_source_file (post-dedup - shows how redundant master_sentence really was):")
print(df["orig_source_file"].value_counts().to_string())

df.insert(0, "concept_id", np.arange(1, len(df) + 1))

print("\nShuffling with the fixed seed.")
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

whole_dataset_df = df.copy()   # full merged/deduped corpus, pre-split - written as-is in Section 9

# Sanity-check the encoding repair from Section 2: scan for mojibake byte patterns that
# show up when this same text is mis-decoded as Latin-1 (curly quotes/apostrophes turning
# into bare "\u00e2" sequences).
_mojibake_pat = r"\u00c3.|\u00e2\u20ac.|\u00e2(?![a-zA-Z])"
_suspect = whole_dataset_df["Ekegusii"].astype(str).str.contains(_mojibake_pat, regex=True, na=False)
print(f"\nrows with suspected mojibake in Ekegusii after encoding repair: {int(_suspect.sum()):,} "
      f"(expected: a small residual from corruption that predates this pipeline, not fixable)")


De-duplication
----------------------------------------------------------------------
Removed 41,704 exact duplicate (English, Ekegusii) pairs.
Rows remaining: 87,453

PSA rows      : 21,362
Non-PSA rows  : 66,091

Rows by category:
category
scripture     65564
psa           21362
dictionary      316
storybook       211

Rows by orig_source_file (post-dedup - shows how redundant master_sentence really was):
orig_source_file
Bible_engswguz                     65528
kenyan_psa_multilingual_dataset    17197
psa_ke_train                        3509
psa_ke_test                          573
MixGenre_EngSwaGuz                   411
master_sentence_EgSwGuzcorpus        235

Shuffling with the fixed seed.

rows with suspected mojibake in Ekegusii after encoding repair: 34 (expected: a small residual from corruption that predates this pipeline, not fixable)


## 5. Three-part test construction: PSA -> Bible -> general Ekegusii

### Method
Built in this exact order, each set removed from the shared pool before the next is constructed:

1. **PSA test** (2,000 target: 400/domain x 5 domains) - candidates restricted to
   `psa_ke_test`/`psa_ke_train` only (native-speaker content), unflagged, length-gated.
2. **Bible test** (2,000, random) - candidates restricted to genuine scripture (folktales excluded).
3. **General Ekegusii test** - four slices, each a percentage of *that source's own remaining
   pool* (bible 5%, storybook 25%, PSA 10% - this slice *does* draw from `kenyan_psa_multilingual`
   as well as the `psa_ke` leftover - dictionary/`lughayangu` 25%). No fixed total.

The length gate is applied to each candidate pool *before* sampling, not after, so the requested
row counts (2,000 / 2,000) are guaranteed achievable rather than falling short post-hoc.

### Rationale
- **Order matters**: general Ekegusii is built last specifically so its percentages are computed
  off what's left once the PSA and Bible tests have already claimed their rows - the three test
  sets are disjoint by construction rather than by a de-duplication pass afterward.
- **PSA test restricted to native-speaker files only**: `kenyan_psa_multilingual` is 73%
  LLM-generated; a "ground truth" test set built from it would partly be testing agreement with
  another model's output, not real Ekegusii. The general test's PSA slice is the one exception,
  by explicit choice - it accepts that tradeoff for a larger, differently-sourced sample and tags
  `provenance` (`real`/`synthetic`) per row so it can be filtered later if a real-only cut is
  needed.
- **Agriculture will very likely fall short of 400.** Of 501 native-speaker Agriculture rows, 145
  (29%) are flagged `quality_flags == "document_extract"` - text pulled from a longer report
  rather than authored as a PSA - a far higher flag rate than the other four domains. Excluding
  them (the same rule applied everywhere) leaves only ~348 clean candidates. This notebook takes
  every eligible row for a short domain rather than loosening the quality bar or hard-failing.

In [36]:
def length_gate_mask(frame, col_a="English", col_b="Ekegusii"):
    wa = frame[col_a].astype(str).str.split().str.len()
    wb = frame[col_b].astype(str).str.split().str.len()
    return wa.between(MIN_WORDS, MAX_WORDS) & wb.between(MIN_WORDS, MAX_WORDS)


pool = df.copy()

print("Test-set construction (PSA -> Bible -> general Ekegusii)")
print("-" * 70)

# --- 5a. PSA test ------------------------------------------------------------
psa_test_candidates = pool[
    (pool["PSA_non_PSA"] == "PSA")
    & (pool["orig_source_file"].isin(["psa_ke_test", "psa_ke_train"]))
    & (pool["quality_flag"].isna())
    & (pool["psa_domain"].isin(CANONICAL_DOMAINS))
]
psa_test_candidates = psa_test_candidates[length_gate_mask(psa_test_candidates)]
print("PSA-test candidate pool by domain (psa_ke_test+train, unflagged, length-gated):")
print(psa_test_candidates["psa_domain"].value_counts().to_string())
counts = psa_test_candidates["psa_domain"].value_counts()
short = counts[counts < PSA_TEST_PER_DOMAIN]
if len(short):
    print(f"\nNOTE: {list(short.index)} fall short of {PSA_TEST_PER_DOMAIN}/domain after excluding "
          f"flagged rows - taking every eligible row for those domains instead of erroring out.")

psa_test_parts = [
    psa_test_candidates[psa_test_candidates["psa_domain"] == dom].sample(
        n=min(PSA_TEST_PER_DOMAIN, counts.get(dom, 0)), random_state=SEED)
    for dom in CANONICAL_DOMAINS
]
psa_test_df = pd.concat(psa_test_parts)
pool = pool.drop(psa_test_df.index)
print(f"\nPSA test set: {len(psa_test_df):,} rows (target {PSA_TEST_PER_DOMAIN}/domain x "
      f"{len(CANONICAL_DOMAINS)} domains = {PSA_TEST_TOTAL:,}; actual per-domain counts below)")
print(psa_test_df["psa_domain"].value_counts().to_string())

# --- 5b. Bible test -----------------------------------------------------------
bible_test_candidates = pool[pool["category"] == "scripture"]
bible_test_candidates = bible_test_candidates[length_gate_mask(bible_test_candidates)]
assert len(bible_test_candidates) >= BIBLE_TEST_TOTAL
bible_test_df = bible_test_candidates.sample(n=BIBLE_TEST_TOTAL, random_state=SEED)
pool = pool.drop(bible_test_df.index)
print(f"\nBible test set: {len(bible_test_df):,} rows (random, scripture only)")

# --- 5c. General Ekegusii test -------------------------------------------------
def sample_frac(frame, frac, label):
    frame = frame[length_gate_mask(frame)]
    n = round(len(frame) * frac)
    sampled = frame.sample(n=n, random_state=SEED)
    print(f"  {label:12s}: {frac:.0%} of {len(frame):,} eligible rows -> {len(sampled):,}")
    return sampled


print("\nGeneral Ekegusii test set (% of each source's own remaining pool):")
scripture_remaining = pool[pool["category"] == "scripture"]
storybook_pool = pool[pool["category"] == "storybook"]
psa_remaining = pool[(pool["PSA_non_PSA"] == "PSA") & (pool["quality_flag"].isna())]
dictionary_pool = pool[pool["category"] == "dictionary"]

general_bible_slice = sample_frac(scripture_remaining, GENERAL_TEST_FRACS["bible"], "bible")
general_story_slice = sample_frac(storybook_pool, GENERAL_TEST_FRACS["storybook"], "storybook")
general_psa_slice = sample_frac(psa_remaining, GENERAL_TEST_FRACS["psa"], "psa")
general_dict_slice = sample_frac(dictionary_pool, GENERAL_TEST_FRACS["dictionary"], "dictionary")

general_test_df = pd.concat(
    [general_bible_slice, general_story_slice, general_psa_slice, general_dict_slice])
pool = pool.drop(general_test_df.index)
print(f"\nGeneral Ekegusii test set: {len(general_test_df):,} rows total")


Test-set construction (PSA -> Bible -> general Ekegusii)
----------------------------------------------------------------------
PSA-test candidate pool by domain (psa_ke_test+train, unflagged, length-gated):
psa_domain
Education      1153
Security        948
Health          865
Governance      480
Agriculture     348

NOTE: ['Agriculture'] fall short of 400/domain after excluding flagged rows - taking every eligible row for those domains instead of erroring out.

PSA test set: 1,948 rows (target 400/domain x 5 domains = 2,000; actual per-domain counts below)
psa_domain
Education      400
Health         400
Governance     400
Security       400
Agriculture    348

Bible test set: 2,000 rows (random, scripture only)

General Ekegusii test set (% of each source's own remaining pool):
  bible       : 5% of 63,320 eligible rows -> 3,166
  storybook   : 25% of 169 eligible rows -> 42
  psa         : 10% of 19,084 eligible rows -> 1,908
  dictionary  : 25% of 311 eligible rows -> 78

General 

## 6. Validation decomposition (general vs. PSA)

### Method
From whatever remains in the pool after all three test sets are removed, sample `VALIDATION_FRAC`
(10%) stratified by `PSA_non_PSA`, then split that validation slice by the same label into two
named files: `validation_general` and `validation_psa`.

### Rationale (unchanged from the original notebook - it still applies here)
A PSA-only validation set would bring the model-selection signal close to the test set's own
distribution - an indirect leak, since a model implicitly selected for PSA-like validation
performance is no longer being evaluated on a wholly independent test set. An undifferentiated
blended validation set is defensible but PSA rows are a small minority of it, so a single blended
validation loss is dominated by Bible-domain examples and barely reacts to PSA-specific progress.
Splitting the same validation slice into two labeled files, at no additional data cost, gives two
independent signals during training: is it forgetting general Ekegusii, and is it actually
improving on PSAs.

In [37]:
def stratified_split(frame, frac, stratify_col, seed):
    '''Stand-in for sklearn's train_test_split(..., stratify=...): sample `frac` from each
    stratum independently so both halves keep the stratum's proportional share.'''
    held_parts, rest_parts = [], []
    for _, group in frame.groupby(stratify_col):
        n_held = round(len(group) * frac)
        held = group.sample(n=n_held, random_state=seed)
        rest = group.drop(held.index)
        held_parts.append(held)
        rest_parts.append(rest)
    return pd.concat(rest_parts), pd.concat(held_parts)


print("Validation split (from the post-test remainder)")
print("-" * 70)
train_df, val_df = stratified_split(pool, VALIDATION_FRAC, "PSA_non_PSA", SEED)
val_general_df = val_df[val_df["PSA_non_PSA"] == "Non-PSA"].reset_index(drop=True)
val_psa_df = val_df[val_df["PSA_non_PSA"] == "PSA"].reset_index(drop=True)
train_df = train_df.reset_index(drop=True)
psa_test_df = psa_test_df.reset_index(drop=True)
bible_test_df = bible_test_df.reset_index(drop=True)
general_test_df = general_test_df.reset_index(drop=True)

print(f"train              : {len(train_df):,} rows")
print(f"validation_general : {len(val_general_df):,} rows")
print(f"validation_psa     : {len(val_psa_df):,} rows")
print(f"test_psa           : {len(psa_test_df):,} rows")
print(f"test_bible         : {len(bible_test_df):,} rows")
print(f"test_general       : {len(general_test_df):,} rows")
_sum = (len(train_df) + len(val_general_df) + len(val_psa_df) + len(psa_test_df)
        + len(bible_test_df) + len(general_test_df))
print(f"sum                : {_sum:,}  (whole_dataset: {len(whole_dataset_df):,})")
assert _sum == len(whole_dataset_df), (
    "partition rows don't sum to the whole dataset - a row leaked or was dropped silently")
print("sanity check passed: every partition sums back to the whole deduped dataset")


Validation split (from the post-test remainder)
----------------------------------------------------------------------
train              : 70,480 rows
validation_general : 6,080 rows
validation_psa     : 1,751 rows
test_psa           : 1,948 rows
test_bible         : 2,000 rows
test_general       : 5,194 rows
sum                : 87,453  (whole_dataset: 87,453)
sanity check passed: every partition sums back to the whole deduped dataset


## 7. Curriculum construction: Stage 1, Stage 2, Mixed

### Method
Unchanged from the original notebook. Three training sets, built from the **train partition
only**:

| Run | Constructed from | Role |
|---|---|---|
| **Stage 1** | Non-PSA rows in train | General Ekegusii competence - the baseline |
| **Stage 2** | PSA rows (upsampled x`PSA_UPSAMPLE`) + a `REPLAY_FRACTION` slice of Stage 1 | PSA-register adaptation, anchored against forgetting |
| **Mixed** | Stage 1 union upsampled PSA, one pass | Control: does staged ordering matter beyond just including the same data? |

Each row expands into up to two directional records: English->Ekegusii always, Kiswahili->Ekegusii
whenever the Kiswahili field is populated.

### Rationale (unchanged)
PSA rows are a small minority of train (a direct consequence of the PSA test set drawing
disproportionately from the smaller native-speaker PSA pool) - upsampling by physical repetition,
rather than a sampler weight, keeps Stage 2's reported composition exactly what the model actually
trains on. Replay anchors against catastrophic forgetting after specializing on a narrow domain.
The Mixed control isolates whether the staged *ordering* contributes anything beyond the data
itself.

In [38]:
def expand(frame, corpus_label):
    '''Up to 2 directional records per row: English->Ekegusii always, Kiswahili->Ekegusii
    only when the Kiswahili field is populated.'''
    out = []
    for r in frame.itertuples(index=False):
        guz = r.Ekegusii
        if pd.isna(guz) or not str(guz).strip():
            continue
        guz = str(guz).strip()
        # psa_domain is a real string for PSA rows but NaN (a float) for every
        # non-PSA row - and stage2/mixed/test all mix both kinds of row in one
        # file. datasets.Dataset.from_list() -> PyArrow needs one consistent
        # type per column and raises "Expected bytes, got a 'float' object" if
        # a column mixes str and float NaN; None is a null Arrow can unify with
        # str in the same column, so convert here rather than downstream.
        psa_domain = None if pd.isna(r.psa_domain) else r.psa_domain
        extra = {"psa_domain": psa_domain, "category": r.category, "provenance": r.provenance}
        eng = r.English
        if not pd.isna(eng) and str(eng).strip():
            out.append({"src_lang": ENG, "tgt_lang": GUZ, "src": str(eng).strip(), "tgt": guz,
                        "corpus": corpus_label, "domain": r.dataset_origin,
                        "concept_id": int(r.concept_id), **extra})
        swh = r.Kiswahili
        if not pd.isna(swh) and str(swh).strip():
            out.append({"src_lang": SWH, "tgt_lang": GUZ, "src": str(swh).strip(), "tgt": guz,
                        "corpus": corpus_label, "domain": r.dataset_origin,
                        "concept_id": int(r.concept_id), **extra})
    return out


def dedupe_records(rows):
    seen, out = set(), []
    for r in rows:
        key = (r["src_lang"], r["tgt_lang"], r["src"].lower(), r["tgt"].lower())
        if key not in seen:
            seen.add(key)
            out.append(r)
    return out


print("Curriculum construction (Stage 1 / Stage 2 / Mixed)")
print("-" * 70)
train_nonpsa = train_df[train_df["PSA_non_PSA"] == "Non-PSA"]
train_psa = train_df[train_df["PSA_non_PSA"] == "PSA"]

stage1 = dedupe_records(expand(train_nonpsa, "non_psa"))
psa_pool = dedupe_records(expand(train_psa, "psa"))

rng = np.random.default_rng(SEED)
psa_up = psa_pool * PSA_UPSAMPLE
n_replay = int(len(psa_up) * REPLAY_FRACTION / (1 - REPLAY_FRACTION))
replay_idx = rng.choice(len(stage1), size=min(n_replay, len(stage1)), replace=False)
replay = [dict(stage1[i], corpus="replay") for i in replay_idx]

stage2 = psa_up + replay
mixed = stage1 + psa_up

print(f"Stage 1  {len(stage1):>7,}   general Ekegusii (Non-PSA rows from train, both directions)")
print(f"Stage 2  {len(stage2):>7,}   {len(psa_up):,} PSA (x{PSA_UPSAMPLE}) + {len(replay):,} replay "
      f"({100*len(replay)/len(stage2):.0f}% of Stage 2)")
print(f"Mixed    {len(mixed):>7,}   Stage 1 union upsampled PSA, single training pass")


Curriculum construction (Stage 1 / Stage 2 / Mixed)
----------------------------------------------------------------------
Stage 1  104,234   general Ekegusii (Non-PSA rows from train, both directions)
Stage 2  164,757   123,568 PSA (x4) + 41,189 replay (25% of Stage 2)
Mixed    227,802   Stage 1 union upsampled PSA, single training pass


## 8. Held-out set construction: length gating and leakage verification

### Method
`validation_general`, `validation_psa`, `test_psa`, `test_bible`, and `test_general` are each
expanded into directional records (same `expand()` as train), filtered to the 2-60 word length
gate on both sides, de-duplicated, then checked against every source-text key in the train pool
(`stage1 u psa_pool`) - any record whose source text already appears in train is dropped.

### Rationale
Length gating is eval-only: the PSA corpus contains long document-style extracts that remain
valid training signal but would let a handful of long documents dominate a corpus-level chrF/BLEU
score, measuring document-length handling rather than sentence-level translation quality.

**A concrete, verified leak this check catches that Section 4's row-level dedup can't:**
`Bible_engswguz`'s `Trilingual-Bible` and `bible_ebible` sources are two *different English* Bible
translations that share the *same* underlying Swahili text for ~90% of verses (often the same
Ekegusii too) - e.g. "They traveled from Haradah, and encamped in Makheloth." (`Trilingual-Bible`)
vs. "They set out from Haradah and camped at Makheloth." (`bible_ebible`), both paired with the
*identical* Swahili and Ekegusii translation. Section 4's dedup keys on (English, Ekegusii), so
both survive as distinct rows - but if one lands in train and the other in an eval set, the
Swahili->Ekegusii direction is leaked even though the English differs. Expect ~30-35% of
Swahili-direction eval candidates to be dropped here for exactly this reason - that's this check
doing its job, not a bug.

In [39]:
def usable(r):
    ns, nt = len(r["src"].split()), len(r["tgt"].split())
    return MIN_WORDS <= ns <= MAX_WORDS and MIN_WORDS <= nt <= MAX_WORDS


def build_eval_set(frame, corpus_label):
    return [r for r in expand(frame, corpus_label) if usable(r)]


train_keys = {(r["src_lang"], r["tgt_lang"], r["src"].lower()) for r in stage1 + psa_pool}


def drop_leaks(rows, label):
    clean = [r for r in rows if (r["src_lang"], r["tgt_lang"], r["src"].lower()) not in train_keys]
    if len(clean) != len(rows):
        leaked = [r for r in rows if (r["src_lang"], r["tgt_lang"], r["src"].lower()) in train_keys]
        by_dir = pd.Series([r["src_lang"] for r in leaked]).value_counts().to_dict()
        print(f"  Dropped {len(rows) - len(clean)} leaked record(s) from {label}  "
              f"(by direction: {by_dir})")
    return clean


print("Held-out set construction (length gate + leakage check)")
print("-" * 70)
eval_specs = [
    (val_general_df, "validation_general"), (val_psa_df, "validation_psa"),
    (psa_test_df, "test_psa"), (bible_test_df, "test_bible"),
    (general_test_df, "test_general"),
]
eval_rows = {}
for frame, label in eval_specs:
    rows = dedupe_records(build_eval_set(frame, label))
    rows = drop_leaks(rows, label)
    eval_rows[label] = rows
    print(f"  {label:20s} {len(rows):>7,} directional records  (from {len(frame):,} rows)")

test_rows_combined = eval_rows["test_psa"] + eval_rows["test_bible"] + eval_rows["test_general"]


Held-out set construction (length gate + leakage check)
----------------------------------------------------------------------
  Dropped 4204 leaked record(s) from validation_general  (by direction: {'swh_Latn': 4179, 'eng_Latn': 25})
  validation_general     7,851 directional records  (from 6,080 rows)
  Dropped 24 leaked record(s) from validation_psa  (by direction: {'eng_Latn': 21, 'swh_Latn': 3})
  validation_psa         3,405 directional records  (from 1,751 rows)
  Dropped 89 leaked record(s) from test_psa  (by direction: {'eng_Latn': 86, 'swh_Latn': 3})
  test_psa               3,020 directional records  (from 1,948 rows)
  Dropped 1419 leaked record(s) from test_bible  (by direction: {'swh_Latn': 1413, 'eng_Latn': 6})
  test_bible             2,576 directional records  (from 2,000 rows)
  Dropped 2264 leaked record(s) from test_general  (by direction: {'swh_Latn': 2229, 'eng_Latn': 35})
  test_general           8,033 directional records  (from 5,194 rows)


## 9. Serialization and export

### Method
Two representations, matching what `data_io.py` already expects:

- **Directional `.jsonl` files**: `stage1.jsonl`, `stage2.jsonl`, `mixed.jsonl`,
  `validation_general.jsonl`, `validation_psa.jsonl` - plus `test_psa.jsonl`, `test_bible.jsonl`,
  `test_general.jsonl` individually (for per-category evaluation), and `test.jsonl` as their union
  (tagged distinctly via each record's `corpus` field), so `data_io.load_curriculum` - which
  hard-codes a single `"test"` name - loads correctly with zero changes.
- **Flat, row-level `.csv` files**: `train.csv`, `validation_general.csv`, `validation_psa.csv`,
  `test_psa.csv`, `test_bible.csv`, `test_general.csv`, and a combined `test.csv`.
- **`whole_dataset.csv`/`.parquet`**: the full merged, deduplicated corpus, pre-split.
- **`mixture.json`**: every partition/curriculum size, the three test sets' internal breakdowns,
  and every hyperparameter used, so any output file's composition can be audited without
  re-running this notebook.

In [40]:
print("Writing output to", DATA_DIR)
print("-" * 70)


def write_jsonl(rows, name):
    path = DATA_DIR / f"{name}.jsonl"
    with open(path, "w", encoding="utf-8") as fh:
        for r in rows:
            fh.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"  {len(rows):>7,} rows -> {path.name}")


print("Directional training/evaluation files:")
for rows, name in [
    (stage1, "stage1"), (stage2, "stage2"), (mixed, "mixed"),
    (eval_rows["validation_general"], "validation_general"),
    (eval_rows["validation_psa"], "validation_psa"),
    (eval_rows["test_psa"], "test_psa"), (eval_rows["test_bible"], "test_bible"),
    (eval_rows["test_general"], "test_general"),
    (test_rows_combined, "test"),
]:
    write_jsonl(rows, name)

print("\nFlat row-level files (pre-expansion, original wide schema):")
flat_specs = [
    (train_df, "train"), (val_general_df, "validation_general"),
    (val_psa_df, "validation_psa"), (psa_test_df, "test_psa"),
    (bible_test_df, "test_bible"), (general_test_df, "test_general"),
]
for d, name in flat_specs:
    path = DATA_DIR / f"{name}.csv"
    d.to_csv(path, index=False)
    print(f"  {len(d):>7,} rows -> {path.name}")

test_flat_combined = pd.concat([psa_test_df, bible_test_df, general_test_df])
test_flat_combined.to_csv(DATA_DIR / "test.csv", index=False)
print(f"  {len(test_flat_combined):>7,} rows -> test.csv  (union of test_psa/test_bible/test_general)")

print("\nWhole dataset (pre-split - merged, shuffled, deduplicated, all rows):")
whole_dataset_df.to_csv(DATA_DIR / "whole_dataset.csv", index=False)
whole_dataset_df.to_parquet(DATA_DIR / "whole_dataset.parquet", index=False)
print(f"  {len(whole_dataset_df):>7,} rows -> whole_dataset.csv / .parquet")

summary = {
    "seed": SEED,
    "whole_dataset_rows": len(whole_dataset_df),
    "row_level_partition": {
        "train": len(train_df), "validation_general": len(val_general_df),
        "validation_psa": len(val_psa_df), "test_psa": len(psa_test_df),
        "test_bible": len(bible_test_df), "test_general": len(general_test_df),
    },
    "directional_record_counts": {
        "stage1": len(stage1), "stage2": len(stage2), "mixed": len(mixed),
        "validation_general": len(eval_rows["validation_general"]),
        "validation_psa": len(eval_rows["validation_psa"]),
        "test_psa": len(eval_rows["test_psa"]), "test_bible": len(eval_rows["test_bible"]),
        "test_general": len(eval_rows["test_general"]), "test": len(test_rows_combined),
    },
    "psa_test_per_domain": psa_test_df["psa_domain"].value_counts().to_dict(),
    "general_test_slice_sizes": {
        "bible": len(general_bible_slice), "storybook": len(general_story_slice),
        "psa": len(general_psa_slice), "dictionary": len(general_dict_slice),
    },
    "hyperparameters": {
        "psa_upsample": PSA_UPSAMPLE, "replay_fraction": REPLAY_FRACTION,
        "min_words": MIN_WORDS, "max_words": MAX_WORDS,
        "psa_test_per_domain": PSA_TEST_PER_DOMAIN, "bible_test_total": BIBLE_TEST_TOTAL,
        "general_test_fracs": GENERAL_TEST_FRACS, "validation_frac": VALIDATION_FRAC,
        "directions": ["eng_Latn->guz_Latn", "swh_Latn->guz_Latn"],
    },
}
with open(DATA_DIR / "mixture.json", "w", encoding="utf-8") as fh:
    json.dump(summary, fh, indent=2, ensure_ascii=False)
print("\nWrote mixture.json:")
print(json.dumps(summary, indent=2, ensure_ascii=False))


Writing output to /home/jovyan/PSA/Final_Training/data
----------------------------------------------------------------------
Directional training/evaluation files:
  104,234 rows -> stage1.jsonl
  164,757 rows -> stage2.jsonl
  227,802 rows -> mixed.jsonl
    7,851 rows -> validation_general.jsonl
    3,405 rows -> validation_psa.jsonl
    3,020 rows -> test_psa.jsonl
    2,576 rows -> test_bible.jsonl
    8,033 rows -> test_general.jsonl
   13,629 rows -> test.jsonl

Flat row-level files (pre-expansion, original wide schema):
   70,480 rows -> train.csv
    6,080 rows -> validation_general.csv
    1,751 rows -> validation_psa.csv
    1,948 rows -> test_psa.csv
    2,000 rows -> test_bible.csv
    5,194 rows -> test_general.csv
    9,142 rows -> test.csv  (union of test_psa/test_bible/test_general)

Whole dataset (pre-split - merged, shuffled, deduplicated, all rows):
   87,453 rows -> whole_dataset.csv / .parquet

Wrote mixture.json:
{
  "seed": 42,
  "whole_dataset_rows": 87453,
  "

## 10. Cross-architecture portability notes

Unchanged from the original notebook - these are properties of the training script, not the data,
so noted here rather than implemented:

- **mBART-50** uses its own code vocabulary (`en_XX`, `sw_KE`, a newly-added Ekegusii token) and
  lacks a close Bantu-language embedding source comparable to NLLB's Kikuyu; Swahili (`sw_KE`) is
  the nearest available initialization target. `src_lang`/`tgt_lang` would need remapping at load
  time.
- **mT5** has no language-identifier tokens at all; its expected input is an instruction prefix,
  e.g. `f"translate English to Ekegusii: {src}"`. The `src`/`tgt` text fields here are reusable
  directly - only the framing around them changes.

**On translation direction**: every record here is *into* Ekegusii only (English->Ekegusii,
Kiswahili->Ekegusii) - `tgt_lang` is always `guz_Latn`. `train.py`'s current design fixes
`tgt_lang` per training run (`tokenizer.tgt_lang = cfg["tgt_code"]`), not per row, so an
Ekegusii->English/Kiswahili direction isn't just a data change - `train.py` would need to vary
`tgt_lang` per record too. That's out of scope here; flagging it since it came up when this data
was scoped.

## 11. Reproducibility summary

In [41]:
import sys, datetime

print("Reproducibility record")
print("=" * 60)
print(f"Generated at        : {datetime.datetime.now().isoformat(timespec='seconds')}")
print(f"Random seed         : {SEED}")
print(f"Python              : {sys.version.split()[0]}")
print(f"pandas              : {pd.__version__}")
print(f"numpy               : {np.__version__}")
print("-" * 60)
print(f"whole dataset       : {len(whole_dataset_df):,} rows")
print(f"train               : {len(train_df):,} rows")
print(f"validation_general  : {len(val_general_df):,} rows")
print(f"validation_psa      : {len(val_psa_df):,} rows")
print(f"test_psa            : {len(psa_test_df):,} rows")
print(f"test_bible          : {len(bible_test_df):,} rows")
print(f"test_general        : {len(general_test_df):,} rows")
print(f"stage1 (directional): {len(stage1):,} records")
print(f"stage2 (directional): {len(stage2):,} records")
print(f"mixed  (directional): {len(mixed):,} records")


Reproducibility record
Generated at        : 2026-09-05T01:21:54
Random seed         : 42
Python              : 3.11.6
pandas              : 2.1.1
numpy               : 1.26.4
------------------------------------------------------------
whole dataset       : 87,453 rows
train               : 70,480 rows
validation_general  : 6,080 rows
validation_psa      : 1,751 rows
test_psa            : 1,948 rows
test_bible          : 2,000 rows
test_general        : 5,194 rows
stage1 (directional): 104,234 records
stage2 (directional): 164,757 records
mixed  (directional): 227,802 records
